In [1]:
"""
Simple Random Forest Example
=============================
Random Forest for Placement Prediction
"""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42

# ---------------------------------------------------------------------
# 1. Load the placement dataset
# ---------------------------------------------------------------------

df = pd.read_csv("/content/placement_predict_50k.csv")

print("Sample of the dataset:")
print(df.head())

print(f"\nDataset shape: {df.shape}")
print(f"\nClass balance:")
print(df["PlacementStatus"].value_counts())

TARGET = "PlacementStatus"

# Drop columns that are not useful predictors
DROP_COLS = []

# ---------------------------------------------------------------------
# 2. Handle missing values
# ---------------------------------------------------------------------

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Convert categorical columns to numerical values
df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

# ---------------------------------------------------------------------
# 3. Select features and target
# ---------------------------------------------------------------------

X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

feature_names = X.columns.tolist()

# ---------------------------------------------------------------------
# 4. Train / test split
# ---------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# ---------------------------------------------------------------------
# 5. Train a Random Forest
# ---------------------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)

# ---------------------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------------------

y_pred = rf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"\nOOB score: {rf.oob_score_:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

# ---------------------------------------------------------------------
# 7. Feature importances
# ---------------------------------------------------------------------

print("Feature importances:")

for name, importance in sorted(
    zip(feature_names, rf.feature_importances_),
    key=lambda x: -x[1]
):
    print(f"  {name}: {importance:.4f}")

Sample of the dataset:
   StudentID  Gender       City CollegeTier      Stream Specialisation Hostel  \
0          1  Female      Delhi       Tier3          IT    DataScience    Yes   
1          2    Male    Chennai       Tier2         ECE             AI    Yes   
2          3  Female  Hyderabad       Tier3         ECE     Networking     No   
3          4  Female     Jaipur       Tier3         ECE       Embedded     No   
4          5    Male  Ahmedabad       Tier3  Mechanical    DataScience     No   

  HistoryOfBacklogs  SGPA_Sem1  SGPA_Sem2  ...  Certifications  Publications  \
0               Yes       6.39       6.84  ...               1             0   
1                No       5.95       6.74  ...               2             0   
2                No       7.13       8.11  ...               2             1   
3                No       9.37       9.97  ...               6             2   
4                No       8.25       8.99  ...               4             2   

   Aptitu